# APIM ❤️ 3rd Party models

## Gemini Models lab
![flow](../../images/gemini-models.gif)

Playground to try Gemini Models served trough the AI Gateway. This lab creates two API's: one with native Gemini API compatibility and another one with OpenAI compatibility. 

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- [Gemini API Key from Google](https://aistudio.google.com/apikey)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`... 


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the models and versions according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "westeurope"

# This lab attaches to a pre-existing, shared APIM instance instead of deploying
# its own — see main.bicep / gemini-shared-resources.bicep for details. The shared
# instance's own Log Analytics workspace (workspace-pdcibwky2f5ms) already receives
# every API's gateway/LLM logs, so this lab does not create its own.
shared_apim_name = "apim-shared-pdcibwky2f5ms"
shared_apim_resource_group_name = "rg-shared-apim-gateway-V2"
shared_log_analytics_workspace_name = "workspace-pdcibwky2f5ms"

apim_subscriptions_config = [{"name": "subscription1", "displayName": "Gemini Models - Subscription 1"},
                             {"name": "subscription2", "displayName": "Gemini Models - Subscription 2"},
                             {"name": "subscription3", "displayName": "Gemini Models - Subscription 3"}]

# IMPORTANT — Gemini's free-tier quota (20 requests/day) is capped per Google
# Cloud project + model, NOT per Azure APIM subscription. If all 3 subscriptions
# above forward to the SAME Google API key, they drain ONE shared 20/day quota
# and you hit 429 RESOURCE_EXHAUSTED almost immediately during the load test.
#
# To give each subscription its OWN independent 20/day quota, generate a
# separate API key for EACH subscription from a DIFFERENT Google Cloud project:
#   1. Go to https://aistudio.google.com/apikey
#   2. Click "Create API key" -> "Create API key in new project" (repeat for
#      each subscription, picking/creating a different project each time)
# If you only have one key, you can repeat it 3x below — but you'll be back to
# sharing a single 20/day quota across all subscriptions.
gemini_api_keys_by_subscription = {
    "subscription1": "YOU-APIS-KEY",
    "subscription2": "YOU-APIS-KEY",
    "subscription3": "YOU-APIS-KEY"
}

openai_compatible_api_path = "gemini-models-openaicompatible"  # path to the inference API in the APIM service with OpenAI compatibility
gemini_inference_api_path = "gemini-models-geminiapi"  # path to the inference API in the APIM service with Google GenAI compatibility
gemini_api_url = "https://generativelanguage.googleapis.com"  # replace with your Gemini API URL
gemini_api_version = "v1beta"
gemini_model_name = "gemini-3-flash-preview"  # replace with your desired Gemini model name
gemini_api_key = gemini_api_keys_by_subscription["subscription1"]  # fallback/default key (used only outside subscription context)

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 13:38:33.185787 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [39]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 13:39:17.610302 :6s]
👉🏽 Current user: ycure@coem.co
👉🏽 Tenant ID: 0c67c9dd-067e-4ab7-adc3-eac91ce94463
👉🏽 Subscription ID: efbaff8f-21cc-49db-8141-2caaf996decd


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations. 

In [40]:
# Create the resource group if it doesn't exist. Nothing is actually deployed
# INTO it — it only exists because `az deployment group create` needs some
# resource group as its nominal target. Every real resource this lab creates
# (its 2 APIs, backends, product, subscriptions) is deployed straight into the
# SHARED APIM's resource group instead, via main.bicep -> gemini-shared-resources.bicep
# (scoped explicitly with `scope:`). This lab also creates no Log Analytics
# workspace or App Insights of its own — it reads the shared instance's existing
# central workspace directly for the analytics cell below.
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "openAICompatibleAPIPath": { "value": openai_compatible_api_path },
        "geminiInferenceAPIPath": { "value": gemini_inference_api_path },
        "geminiAPIURL": { "value": gemini_api_url },
        "geminiAPIKey": { "value": gemini_api_key },
        "geminiApiKeysBySubscription": { "value": gemini_api_keys_by_subscription },
        "sharedApimName": { "value": shared_apim_name },
        "sharedApimResourceGroupName": { "value": shared_apim_resource_group_name },
        "sharedLogAnalyticsWorkspaceName": { "value": shared_log_analytics_workspace_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

⚙️ Running: az group show --name lab-gemini-models 
👉🏽 Using existing resource group 'lab-gemini-models'
⚙️ Running: az deployment group create --name gemini-models --resource-group lab-gemini-models --template-file main.bicep --parameters params.json 
✅ Deployment 'gemini-models' succeeded ⌚ 13:40:57.189808 :26s]


<a id='3'></a>
### 3️⃣ Get the deployment outputs

We are now at the stage where we only need to retrieve the gateway URL and the subscription before we are ready for testing.

In [41]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key") # default api key to the first subscription key

⚙️ Running: az deployment group show --name gemini-models -g lab-gemini-models 
✅ Retrieved deployment: gemini-models ⌚ 13:41:10.371800 :4s]
👉🏽 Log Analytics Id: 16203b9c-3fb0-4b06-8f6b-114cc352720e
👉🏽 APIM Service Id: /subscriptions/efbaff8f-21cc-49db-8141-2caaf996decd/resourceGroups/rg-shared-apim-gateway-V2/providers/Microsoft.ApiManagement/service/apim-shared-pdcibwky2f5ms
👉🏽 APIM API Gateway URL: https://apim-shared-pdcibwky2f5ms.azure-api.net
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****afd4
👉🏽 Subscription Name: subscription2
👉🏽 Subscription Key: ****25c1
👉🏽 Subscription Name: subscription3
👉🏽 Subscription Key: ****dacf


<a id='requests'></a>
### 🧪 Test the API using a direct HTTP call

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

In [42]:
import json, requests, time
url = f"{apim_resource_gateway_url}/{gemini_inference_api_path}/{gemini_api_version}/models/{gemini_model_name}:generateContent"

contents = {
    "contents": [
      {
        "parts": [
          {
            "text": "Explain how AI works in a few words"
          }
        ]
      }
    ]
  }

# Initialize a session for connection pooling and set any default headers
session = requests.Session()
session.headers.update({
    'x-goog-api-key': api_key,
    'x-user-id': 'alex'
})

try:
    start_time = time.time()
    response = session.post(url, json = contents)
    response_time = time.time() - start_time
    print(f"⌚ {response_time:.2f} seconds")

    utils.print_response_code(response)
    print(f"Response headers: {json.dumps(dict(response.headers), indent = 4)}")

    print(f"{response.text}\n")

finally:
    # Close the session to release the connection
    session.close()


⌚ 4.03 seconds
Response status: 200 - OK
Response headers: {
    "Content-Type": "application/json; charset=UTF-8",
    "Date": "Wed, 09 Sep 2026 18:41:23 GMT",
    "Server": "scaffolding on HTTPServer2",
    "Alt-Svc": "h3=\":443\"; ma=2592000,h3-29=\":443\"; ma=2592000",
    "Content-Encoding": "gzip",
    "Transfer-Encoding": "chunked",
    "Vary": "Origin,X-Origin,Referer",
    "X-Gemini-Service-Tier": "standard",
    "X-XSS-Protection": "0",
    "X-Frame-Options": "SAMEORIGIN",
    "X-Content-Type-Options": "nosniff",
    "Server-Timing": "gfet4t7; dur=2002",
    "x-debug-key-source": "gemini-models-key-subscription1"
}
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "AI finds **patterns in data** to make **predictions** or decisions.",
            "thoughtSignature": "EoYHCoMHARFNMg/1Vr/TzruEfUVd2KpzZ5Mwylld+vUWT+sRmTLD2zVdb873s/RxGOUi+YcAb2OxFZmE/10NbiyEve7q24K1CPulhQ/a9OKk3M39/qH3ocorgJCnb2UbiLyGLyFwc2DP/FZ5GwZo4Lax/hJO8ANCI33TClV

<a id='googlesdk'></a>
### 🧪 Test the inference API with the Google GenAI SDK


In [43]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key,
    http_options=types.HttpOptions(
        base_url=f"{apim_resource_gateway_url}/{gemini_inference_api_path}",
        api_version=gemini_api_version,
    )
)

response = client.models.generate_content(
    model=gemini_model_name, contents="Explain how AI works in a few words"
)
print(response.text)

It learns **patterns** from **data** to make **predictions**.


<a id='sdk'></a>
### 🧪 Test with streaming using the OpenAI Python SDK

Notice that the base_url is appended with `openai` to ensure OpenAI API compatibility.   
With a streaming API call, the response is sent back incrementally in chunks via an [event stream](https://developer.mozilla.org/docs/Web/API/Server-sent_events/Using_server-sent_events#event_stream_format). In Python, you can iterate over these events with a for loop.

In [44]:
import time
from openai import OpenAI
client = OpenAI(
    api_key=api_key, # the api key will be sent as an authorization bearer token. You need to set the 'bearer' enabled in the APIM api settings.
    base_url=f"{apim_resource_gateway_url}/{openai_compatible_api_path}/{gemini_api_version}/openai"
)
response = client.chat.completions.with_raw_response.create(
    model=gemini_model_name,
    messages=[{"role": "user", "content": "Count to 100, with a comma between each number and no newlines. E.g., 1, 2, 3, ..."}],
    stream=True
)

print("headers ", response.headers)

completion = response.parse() 

# create variables to collect the stream of chunks
collected_chunks = []
collected_messages = []
# iterate through the stream of events
for chunk in completion:
    chunk_time = time.time() - start_time  # calculate the time delay of the chunk
    collected_chunks.append(chunk)  # save the event response
    if chunk.choices:
        chunk_message = chunk.choices[0].delta.content  # extract the message
        collected_messages.append(chunk_message)  # save the message
        print(f"Message received {chunk_time:.2f} seconds after request: {chunk_message}")  # print the delay and text
# print the time delay and text received
print(f"Full response received {chunk_time:.2f} seconds after request")  # type: ignore
# clean None in collected_messages
collected_messages = [m for m in collected_messages if m is not None]
full_reply_content = ''.join(collected_messages)
print(f"Full conversation received: {full_reply_content}")




headers  Headers({'content-type': 'text/event-stream', 'date': 'Wed, 09 Sep 2026 18:42:04 GMT', 'server': 'scaffolding on HTTPServer2', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'cache-control': 'private', 'content-encoding': 'gzip', 'expires': 'Wed, 09 Sep 2026 18:42:04 GMT', 'set-cookie': 'NID=534=LwFaVCjYwXi57wUu8MK7qvlRq-CuPrOWeKAUhbQEWfspEqakaVKxEWV2nvGrQVWmo9tiZoqkXucNu7xSL-Tso8oF_oJ_F9Pgv6wi1m7zMShpOcdPGzDD_fR8o8311JOam5uxIBFjt_2Sy1EGb-Kfq4pn91i3cBRlMpy83V1U0XNANoJtmmKRmKV_AX0DpRDRoG-ZAAideySmMQpV6fA9ZNYKS5xp1zgDe7R5wAcfFw; expires=Thu, 11-Mar-2027 18:42:02 GMT; path=/; domain=.generativelanguage.googleapis.com; HttpOnly', 'transfer-encoding': 'chunked', 'vary': 'Origin,X-Origin,Referer', 'p3p': 'CP="This is not a P3P policy! See g.co/p3phelp for more info."', 'x-xss-protection': '0', 'x-frame-options': 'SAMEORIGIN', 'x-content-type-options': 'nosniff', 'server-timing': 'gfet4t7; dur=1643', 'x-debug-key-source': 'gemini-models-key-subscription1'})
Message rece

<a id='sdk'></a>
### 🧪 Execute multiple runs for each subscription using the OpenAI Python SDK

We will send requests for each subscription. Adjust the `sleep_time_ms` and the number of `runs` to your test scenario.

Each subscription now forwards to its own Google API key (configured per-entry in `apim_subscriptions_config` above), so each one has an independent 20-requests/day free-tier quota — 10 runs per subscription stays well under that, no need to reduce it further.

In [45]:
import time
from openai import OpenAI

runs = 3
sleep_time_ms = 100
base_url=f"{apim_resource_gateway_url}/{openai_compatible_api_path}/{gemini_api_version}/openai"

clients = [
    OpenAI(api_key=apim_subscriptions[0].get("key"), base_url=base_url),
    OpenAI(api_key=apim_subscriptions[1].get("key"), base_url=base_url),
    OpenAI(api_key=apim_subscriptions[2].get("key"), base_url=base_url)
]

for i in range(runs):
    print(f"▶️ Run {i+1}/{runs}:")

    for j in range(0, 3):
        response = clients[j].chat.completions.create(
                model=gemini_model_name,
                messages = [
                    {"role": "system", "content": "You are a helpful, professional assistant."},
                    {"role": "user", "content": "Can you tell me the time, please?"}
                ],
                extra_headers = {"x-user-id": "alex"}
            )

        print(f"💬 Subscription {j+1}: {response.choices[0].message.content}")

    print()

    time.sleep(sleep_time_ms/1000)


▶️ Run 1/3:
💬 Subscription 1: Certainly! The current time is 11:39 AM UTC.
💬 Subscription 2: The current time is 2:12 PM UTC on Wednesday, May 22, 2024. 

Please note that this is in Coordinated Universal Time (UTC); if you need the time for a specific city or time zone, feel free to let me know!
💬 Subscription 3: The current time is 11:37 AM.

▶️ Run 2/3:
💬 Subscription 1: It is currently **11:27 AM UTC** on Tuesday, May 21, 2024. 

Since I don't know your specific location, you may want to check your local device or provide your city/timezone for the time in your area!
💬 Subscription 2: It is currently **10:48 AM UTC**. 

To give you the exact local time, please let me know your city or time zone!
💬 Subscription 3: None

▶️ Run 3/3:
💬 Subscription 1: The current time is 10:48 AM UTC. 

Since I don't know your specific location, you may need to adjust that for your local time zone. Would you like me to convert it to a specific time zone for you?
💬 Subscription 2: It is currently 11:42

<a id='kql'></a>
### 🔍 Display LLM logging


In [48]:
import pandas as pd

query = "let llmHeaderLogs = ApiManagementGatewayLlmLog \
        | where DeploymentName != ''; \
        let llmLogsWithSubscriptionId = llmHeaderLogs \
        | join kind=leftouter ApiManagementGatewayLogs on CorrelationId \
        | project \
            SubscriptionId = ApimSubscriptionId, DeploymentName, ModelName, TotalTokens; \
        llmLogsWithSubscriptionId \
        | summarize \
            SumTotalTokens      = sum(TotalTokens) \
        by SubscriptionId, DeploymentName, ModelName"

output = utils.run(f"az monitor log-analytics query -w {log_analytics_id} --analytics-query \"{query}\"", "Retrieved log analytics query output", "Failed to retrieve log analytics query output") 
if output.success and output.json_data:
    table = output.json_data
    display(pd.DataFrame(table))


⚙️ Running: az monitor log-analytics query -w 16203b9c-3fb0-4b06-8f6b-114cc352720e --analytics-query "let llmHeaderLogs = ApiManagementGatewayLlmLog         | where DeploymentName != '';         let llmLogsWithSubscriptionId = llmHeaderLogs         | join kind=leftouter ApiManagementGatewayLogs on CorrelationId         | project             SubscriptionId = ApimSubscriptionId, DeploymentName, ModelName, TotalTokens;         llmLogsWithSubscriptionId         | summarize             SumTotalTokens      = sum(TotalTokens)         by SubscriptionId, DeploymentName, ModelName" 
✅ Retrieved log analytics query output ⌚ 14:08:04.916629 :12s]


,DeploymentName,ModelName,SubscriptionId,SumTotalTokens,TableName
0,gemini-3-flash-preview,gemini-3-flash-preview,gemini-models-subscription1,5922,PrimaryResult
1,gemini-3-flash-preview,gemini-3-flash-preview,gemini-models-subscription2,7061,PrimaryResult
2,gemini-3-flash-preview,gemini-3-flash-preview,gemini-models-subscription3,6235,PrimaryResult
3,gemini-3-flash-preview,,gemini-models-subscription3,0,PrimaryResult
4,DeepSeek-V3.2,deepseek-v3.2,subscription4,7024,PrimaryResult
5,gpt-5-mini,,,2003,PrimaryResult
6,DeepSeek-V3.2,deepseek-v3.2,subscription1,2375,PrimaryResult
7,gpt-5.4-mini,gpt-5.4-mini-2026-03-17,subscription2,1540,PrimaryResult
8,gpt-5.4,gpt-5.4-2026-03-05,subscription3,1979,PrimaryResult
9,gpt-5.4-mini,gpt-5.4-mini-2026-03-17,subscription4,3922,PrimaryResult


<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.